# Fish detector from a pretrained model (like the coral SegFormer) - Colab

Starts from the **Community Fish Detector** (YOLOv12x, pretrained on 1.9 M images / 935 k fish boxes from 17 open
datasets; github.com/filippovarini/community-fish-detector, AGPL). The way the coral model started from EPFL's
Coralscapes SegFormer.

1. **As-is** (no training): scored on validation and test.
2. **Short fine-tune** (~25 min, `FINETUNE_HOURS`) on TrashCan + FathomNet + DeepFish train, then scored the same way.
3. The version with the higher **validation** F1 on TrashCan is kept (test is never used to choose).

TrashCan is **not** in CFD's training data, so `test_trashcan` is a clean test (compare with the current F1 0.229).
CFD **did** see DeepFish (its segmentation subset), so the `test_new` (DeepFish) score may be optimistic.

**Runtime:** T4 GPU. **Needs:** `MyDrive/fish/fish_dataset.zip`. Time: ~10 min setup + ~10 min per evaluation + fine-tune.
**Bring back:** `MyDrive/fish/best_fish_cfd*.pt` (the one printed at the end) and both `fish_cfd*_results.json`.

In [ ]:
!nvidia-smi
!pip install -q -U ultralytics kagglehub

In [ ]:
MODEL = 'cfd-yolov12x-1.00.pt'   # Community Fish Detector, pretrained
IMGSZ = 1024        # the size CFD was trained at
BATCH = 4
EPOCHS = 80
PATIENCE = 20
RUN_NAME = 'cfd_finetune'
FINETUNE_HOURS = 0.42   # ~25 min; set 0 to skip fine-tuning
RESUME = False
FRAME_STEP = 3       # keep every 3rd frame of DeepFish/Brackish videos (neighbours are near-duplicates)
SEED = 42
USE_DEEPFISH = True  # set False to skip a dataset that fails to download or has no YOLO labels
USE_BRACKISH = False # Kaggle copy: label files do not match any frame image
# Only if the class check below cannot find class names: set the YOLO class ids that mean "fish"
DEEPFISH_FISH_IDS = None   # e.g. [0]
BRACKISH_FISH_IDS = None   # e.g. [0, 4]  (fish, small_fish)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile
DRIVE_DIR = '/content/drive/MyDrive/fish'
ZIP = f'{DRIVE_DIR}/fish_dataset.zip'
assert os.path.exists(ZIP), f'Upload fish_dataset.zip to {ZIP}'
if not os.path.exists('/content/fish/processed_fish'):
    with zipfile.ZipFile(ZIP) as z:
        z.extractall('/content/fish')
V1 = '/content/fish/processed_fish'
for s in ('train', 'val', 'test'):
    print('v1', s, len(os.listdir(f'{V1}/{s}/images')), 'images')

## Download DeepFish and Brackish from Kaggle
Public datasets usually download without a login. If it fails, add `KAGGLE_USERNAME` and `KAGGLE_KEY`
(from kaggle.com -> Settings -> API) in Colab **Secrets** (key icon on the left), allow notebook access, and rerun.
Brackish is large (several GB); Colab's disk is enough.

In [ ]:
import kagglehub
try:
    from google.colab import userdata
    for k in ('KAGGLE_USERNAME', 'KAGGLE_KEY'):
        try:
            os.environ[k] = userdata.get(k)
        except Exception:
            pass
except ImportError:
    pass
SOURCES = {}
if USE_DEEPFISH:
    SOURCES['deepfish'] = kagglehub.dataset_download('vencerlanz09/deep-fish-object-detection')
if USE_BRACKISH:
    SOURCES['brackish'] = kagglehub.dataset_download('aalborguniversity/brackish-dataset')
print(SOURCES)

## Inspect the downloads
Finds every image, every YOLO label file and every class-name file, and shows which class ids occur.
The build step stops if it cannot tell which ids are fish, rather than guessing.

In [ ]:
import re, yaml, collections
from pathlib import Path
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp'}

def parse_yolo(path):
    """Return list of (cls, xc, yc, w, h) or None if the file is not a YOLO label file.
    Accepts box lines (cls xc yc w h) and polygon lines (cls x1 y1 x2 y2 ...); a polygon becomes its bounding box."""
    try:
        lines = [l.split() for l in Path(path).read_text(errors='strict').splitlines() if l.strip()]
    except Exception:
        return None
    out = []
    for p in lines:
        try:
            c = int(float(p[0])); v = [float(x) for x in p[1:]]
        except ValueError:
            return None
        if len(v) >= 6 and len(v) % 2 == 0:   # polygon
            if min(v) < -0.01 or max(v) > 1.01:
                return None
            v = [min(max(x, 0.0), 1.0) for x in v]
            xs, ys = v[0::2], v[1::2]
            v = [(min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2, max(xs) - min(xs), max(ys) - min(ys)]
            if v[2] <= 0 or v[3] <= 0:
                continue
        elif len(v) != 4:
            return None
        xc, yc, w, h = v
        if not (0 <= xc <= 1.01 and 0 <= yc <= 1.01 and 0 < w <= 1.01 and 0 < h <= 1.01):
            return None
        out.append((c, xc, yc, w, h))
    return out

def read_names(root):
    found = {}
    for p in Path(root).rglob('*'):
        if p.name in ('data.yaml', 'dataset.yaml') :
            n = (yaml.safe_load(p.read_text()) or {}).get('names')
            if isinstance(n, list): n = dict(enumerate(n))
            if n: found[str(p)] = {int(k): str(v) for k, v in n.items()}
        elif p.suffix == '.names' or p.name in ('classes.txt', 'obj.names'):
            n = [l.strip() for l in p.read_text().splitlines() if l.strip()]
            if n: found[str(p)] = dict(enumerate(n))
    return found

def scan(root):
    root = Path(root)
    images = [p for p in root.rglob('*') if p.suffix.lower() in IMG_EXT]
    labels = {}
    for p in root.rglob('*.txt'):
        if p.name in ('classes.txt', 'README.txt', 'readme.txt'):
            continue
        r = parse_yolo(p)
        if r is not None:
            labels[p] = r
    hist = collections.Counter(c for r in labels.values() for c, *_ in r)
    return images, labels, hist, read_names(root)

SCANS = {}
for src, root in SOURCES.items():
    imgs, labs, hist, names = scan(root)
    SCANS[src] = (imgs, labs, names)
    print(f'== {src}: {len(imgs)} images, {len(labs)} YOLO label files, class-id counts {dict(sorted(hist.items()))}')
    for f, n in names.items():
        print('   names in', f, '->', n)

In [ ]:
NOT_FISH = ('star', 'jelly', 'cray', 'shell')

def fish_ids_for(src, names, override):
    if override is not None:
        return list(override)
    labs = SCANS[src][1]
    if not labs:
        raise SystemExit(f'{src}: no YOLO label files found. Set USE_{src.upper()} = False in the config cell and rerun.')
    used = sorted({c for r in labs.values() for c, *_ in r})
    for n in names.values():
        ids = [k for k, v in n.items() if 'fish' in v.lower() and not any(w in v.lower() for w in NOT_FISH)]
        if ids and set(used) <= set(n):
            return ids
    if used == [0]:   # single-class fish dataset without a names file
        return [0]
    raise SystemExit(f'{src}: cannot tell which class ids are fish (ids used {used}, names {names}). '
                     f'Set {src.upper()}_FISH_IDS in the config cell, or USE_{src.upper()} = False if it has no '
                     f'YOLO labels ({len(labs)} label files found), and rerun.')

OVERRIDES = {'deepfish': DEEPFISH_FISH_IDS, 'brackish': BRACKISH_FISH_IDS}
FISH_IDS = {src: fish_ids_for(src, SCANS[src][2], OVERRIDES[src]) for src in SOURCES}
print('fish class ids:', FISH_IDS)

## Build the merged dataset
Images are linked, not copied. Only images that have a label file are used (an image without a label
file is unknown, not "no fish"). Video id = folder + file name with the trailing frame number removed.

In [ ]:
import hashlib, json, os, shutil
OUT = Path('/content/fish_v2')
if OUT.exists():
    shutil.rmtree(OUT)
for s in ('train', 'val', 'test_trashcan', 'test_new'):
    (OUT / s / 'images').mkdir(parents=True); (OUT / s / 'labels').mkdir(parents=True)

def video_key(img):
    stem = re.sub(r'[-_ .]*(frame)?\d+$', '', img.stem, flags=re.I)
    return stem   # not the folder: Kaggle train/valid folders can hold frames of the same video

def split_of(key):   # deterministic 80/10/10 split by video
    h = int(hashlib.md5(f'{SEED}:{key}'.encode()).hexdigest(), 16) % 100
    return 'train' if h < 80 else ('val' if h < 90 else 'test_new')

def label_for(img, labels_by_path, labels_by_stem):
    for cand in (img.with_suffix('.txt'),
                 Path(str(img.with_suffix('.txt')).replace('/images/', '/labels/'))):
        if cand in labels_by_path:
            return labels_by_path[cand]
    hits = labels_by_stem.get(img.stem, [])
    return labels_by_path[hits[0]] if len(hits) == 1 else None

GROUPS, STATS = {}, collections.Counter()
def put(split, name, img_path, rows, group, src):
    os.symlink(img_path, OUT / split / 'images' / name)
    (OUT / split / 'labels' / (Path(name).stem + '.txt')).write_text(
        ''.join(f'0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n' for xc, yc, w, h in rows))
    GROUPS[f'{split}/{name}'] = group
    STATS[(src, split, 'images')] += 1; STATS[(src, split, 'fish')] += len(rows)

# v1 data: TrashCan / FathomNet, splits kept exactly as before
for s_in, s_out in (('train', 'train'), ('val', 'val'), ('test', 'test_trashcan')):
    for img in sorted(Path(V1, s_in, 'images').iterdir()):
        rows = [r[1:] for r in (parse_yolo(Path(V1, s_in, 'labels', img.stem + '.txt')) or [])]
        m = re.match(r'vid_(\d+)_frame', img.name)
        put(s_out, img.name, img, rows, f'trashcan/{m.group(1)}' if m else f'v1/{img.stem}', 'v1 (TrashCan+FathomNet)')

for src in SOURCES:
    prefix = {'deepfish': 'df', 'brackish': 'br'}[src]
    imgs, labs, _ = SCANS[src]
    by_stem = collections.defaultdict(list)
    for p in labs:
        by_stem[p.stem].append(p)
    videos = collections.defaultdict(list)
    for img in imgs:
        rows = label_for(img, labs, by_stem)
        if rows is not None:
            videos[video_key(img)].append((img, rows))
    sizes = sorted(map(len, videos.values()))
    print(f'{src}: {sum(sizes)} labelled images in {len(videos)} videos, median {sizes[len(sizes) // 2] if sizes else 0} frames/video;',
          'examples:', list(videos)[:3])
    assert len(videos) >= 10, f'{src}: only {len(videos)} video groups found - check video_key() on these file names'
    if sizes[len(sizes) // 2] == 1:
        print(f'WARNING {src}: file names carry no frame numbers, so the split is per image; '
              'near-duplicate frames may land in both train and test. Treat its test score with caution.')
    for key, frames in videos.items():
        split = split_of(f'{src}/{key}')
        for i, (img, rows) in enumerate(sorted(frames, key=lambda t: t[0].name)):
            if i % FRAME_STEP:
                continue
            fish = [r[1:] for r in rows if r[0] in FISH_IDS[src]]
            name = f'{prefix}_{hashlib.md5(str(img).encode()).hexdigest()[:10]}_{img.stem[-40:]}{img.suffix.lower()}'
            put(split, name, img, fish, f'{src}/{key}', src)

(OUT / 'groups.json').write_text(json.dumps(GROUPS))
import pandas as pd
summary = pd.Series(STATS).unstack(level=[1, 2]).fillna(0).astype(int)
print(summary.to_string())
yaml.safe_dump({'path': str(OUT), 'train': 'train/images', 'val': 'val/images', 'test': 'test_trashcan/images',
                'names': {0: 'fish'}}, open(OUT / 'data.yaml', 'w'), sort_keys=False)

## 1. Pretrained model as-is

In [ ]:
!wget -q -nc https://github.com/filippovarini/community-fish-detector/releases/download/cfd-1.00-yolov12x/cfd-yolov12x-1.00.pt
from ultralytics import YOLO
PROJECT = f'{DRIVE_DIR}/runs'
EVAL_WEIGHTS, TAG = 'cfd-yolov12x-1.00.pt', 'fish_cfd_asis'

In [ ]:
import numpy as np
from PIL import Image
EVAL_WEIGHTS = globals().get('EVAL_WEIGHTS') or str(Path(PROJECT) / RUN_NAME / 'weights' / 'best.pt')
TAG = globals().get('TAG') or 'fish_v2'
if Path(EVAL_WEIGHTS).resolve() != Path(f'{DRIVE_DIR}/best_{TAG}.pt').resolve():
    shutil.copyfile(EVAL_WEIGHTS, f'{DRIVE_DIR}/best_{TAG}.pt')
model = YOLO(EVAL_WEIGHTS)
GROUPS = json.loads((OUT / 'groups.json').read_text())

def ul_metrics(split):
    y = OUT / f'eval_{split}.yaml'
    yaml.safe_dump({'path': str(OUT), 'train': 'train/images', 'val': f'{split}/images', 'names': {0: 'fish'}}, open(y, 'w'))
    m = model.val(data=str(y), split='val', imgsz=IMGSZ, batch=BATCH, device=0, plots=False, verbose=False)
    return {'precision': round(float(m.box.mp), 3), 'recall': round(float(m.box.mr), 3),
            'mAP50': round(float(m.box.map50), 3), 'mAP50_95': round(float(m.box.map), 3)}

def gt_boxes(split, name):
    w, h = Image.open(OUT / split / 'images' / name).size
    rows = parse_yolo(OUT / split / 'labels' / (Path(name).stem + '.txt')) or []
    return np.array([[(xc - bw / 2) * w, (yc - bh / 2) * h, (xc + bw / 2) * w, (yc + bh / 2) * h]
                     for _, xc, yc, bw, bh in rows], np.float32).reshape(-1, 4)

def predict(split, names):
    preds = {}
    for i in range(0, len(names), 16):
        batch = [str(OUT / split / 'images' / n) for n in names[i:i + 16]]
        for n, r in zip(names[i:i + 16], model.predict(batch, conf=0.02, imgsz=IMGSZ, device=0, verbose=False)):
            preds[n] = (r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy())
    return preds

def iou(a, b):
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda z: (z[:, 2] - z[:, 0]) * (z[:, 3] - z[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)

def match(gt, boxes, confs, conf):   # greedy, highest confidence first, IoU >= 0.5 (project rule)
    keep = confs >= conf
    boxes, confs = boxes[keep], confs[keep]
    order = np.argsort(-confs)
    if len(gt) == 0:
        return 0, len(order), 0
    ious = iou(boxes, gt) if len(boxes) else np.zeros((0, len(gt)))
    used, tp = set(), 0
    for i in order:
        best_g, best_iou = -1, 0.5
        for g in range(len(gt)):
            if g not in used and ious[i, g] >= best_iou:
                best_g, best_iou = g, ious[i, g]
        if best_g >= 0:
            used.add(best_g); tp += 1
    return tp, len(order) - tp, len(gt) - len(used)

def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0; r = tp / (tp + fn) if tp + fn else 0.0
    return p, r, (2 * p * r / (p + r) if p + r else 0.0)

def per_image(split, names, gts, preds, conf):
    return {n: match(gts[n], *preds[n], conf) for n in names}

def total(per):
    return tuple(int(sum(v[k] for v in per.values())) for k in range(3))

def load(split, keep=lambda n: True):
    names = sorted(n for n in os.listdir(OUT / split / 'images') if keep(n))
    return names, {n: gt_boxes(split, n) for n in names}, predict(split, names)

is_trashcan = lambda n: n.startswith('vid_')
is_new = lambda n: n.startswith(('df_', 'br_'))
RESULTS = {'model': EVAL_WEIGHTS, 'imgsz': IMGSZ, 'run': RUN_NAME, 'matching': 'greedy IoU>=0.5, threshold chosen on val'}
rng = np.random.default_rng(0)
for tag, val_keep, test_split in (('trashcan', is_trashcan, 'test_trashcan'), ('new_datasets', is_new, 'test_new')):
    vn, vg, vp = load('val', val_keep)
    table = []
    for c in np.round(np.arange(0.05, 0.91, 0.05), 2):
        table.append((prf(*total(per_image('val', vn, vg, vp, c)))[2], float(c)))
    best_f1_val, conf = max(table)
    tn, tg, tp_ = load(test_split)
    per = per_image(test_split, tn, tg, tp_, conf)
    p, r, f1 = prf(*total(per))
    vids = collections.defaultdict(list)
    for n in tn:
        vids[GROUPS[f'{test_split}/{n}']].append(n)
    keys = list(vids); boot = []
    for _ in range(1000):
        sample = [n for k in rng.choice(keys, len(keys)) for n in vids[k]]
        boot.append(prf(*(sum(per[n][k] for n in sample) for k in range(3))))
    ci = lambda j: [round(float(np.percentile([b[j] for b in boot], q)), 3) for q in (2.5, 97.5)]
    tp, fp, fn = total(per)
    RESULTS[tag] = {
        'test_split': test_split, 'n_images': len(tn), 'n_videos': len(keys), 'n_fish': tp + fn,
        'conf_chosen_on_val': conf, 'val_f1_at_conf': round(best_f1_val, 3),
        'test': {'precision': round(p, 3), 'recall': round(r, 3), 'f1': round(f1, 3), 'tp': tp, 'fp': fp, 'fn': fn,
                 'precision_ci': ci(0), 'recall_ci': ci(1), 'f1_ci': ci(2)},
        'ultralytics_test': ul_metrics(test_split),
    }
    print(tag, json.dumps(RESULTS[tag], indent=1))

RESULTS['baseline_trashcan_test'] = {'model': 'old 16-class detector, conf 0.18', 'precision': 0.407, 'recall': 0.159, 'f1': 0.229}
json.dump(RESULTS, open(f'{DRIVE_DIR}/{TAG}_results.json', 'w'), indent=2)
print('\nTrashCan test F1: new', RESULTS['trashcan']['test']['f1'], RESULTS['trashcan']['test']['f1_ci'], 'vs baseline 0.229')
print('Saved', f'{DRIVE_DIR}/best_{TAG}.pt', 'and', f'{DRIVE_DIR}/{TAG}_results.json', '-> models/fish_detector/')

In [ ]:
# Image-level "fish present?" accuracy, same format as the coral model's image_level_coral_present
for tag, test_split in (('trashcan', 'test_trashcan'), ('new_datasets', 'test_new')):
    conf = RESULTS[tag]['conf_chosen_on_val']
    names, gts, preds = load(test_split)
    y = np.array([len(gts[n]) > 0 for n in names]); p = np.array([bool((preds[n][1] >= conf).any()) for n in names])
    tp = int((y & p).sum())
    RESULTS[tag]['image_level_fish_present'] = {
        'accuracy': round(float((y == p).mean()), 3), 'precision': round(tp / max(int(p.sum()), 1), 3),
        'recall': round(tp / max(int(y.sum()), 1), 3), 'n_positive': int(y.sum()), 'n_images': len(names),
        'always_majority_baseline': round(float(max(y.mean(), 1 - y.mean())), 3)}
    print(tag, RESULTS[tag]['image_level_fish_present'])
json.dump(RESULTS, open(f'{DRIVE_DIR}/{TAG}_results.json', 'w'), indent=2)
print('updated', f'{DRIVE_DIR}/{TAG}_results.json')

In [ ]:
ASIS = RESULTS

## 2. Short fine-tune (backbone frozen: keeps what CFD learned, trains fast)

In [ ]:
if FINETUNE_HOURS > 0:
    YOLO('cfd-yolov12x-1.00.pt').train(
        data=str(OUT / 'data.yaml'), time=FINETUNE_HOURS, epochs=100, patience=100,
        imgsz=IMGSZ, batch=BATCH, freeze=10,
        optimizer='AdamW', lr0=0.0002, lrf=0.1, warmup_epochs=0, cos_lr=True, close_mosaic=1,
        device=0, seed=SEED, workers=4, cache=False,
        project=PROJECT, name=RUN_NAME, exist_ok=True, plots=False)
EVAL_WEIGHTS, TAG = str(Path(PROJECT) / RUN_NAME / 'weights' / 'best.pt'), 'fish_cfd_finetuned'

In [ ]:
import numpy as np
from PIL import Image
EVAL_WEIGHTS = globals().get('EVAL_WEIGHTS') or str(Path(PROJECT) / RUN_NAME / 'weights' / 'best.pt')
TAG = globals().get('TAG') or 'fish_v2'
if Path(EVAL_WEIGHTS).resolve() != Path(f'{DRIVE_DIR}/best_{TAG}.pt').resolve():
    shutil.copyfile(EVAL_WEIGHTS, f'{DRIVE_DIR}/best_{TAG}.pt')
model = YOLO(EVAL_WEIGHTS)
GROUPS = json.loads((OUT / 'groups.json').read_text())

def ul_metrics(split):
    y = OUT / f'eval_{split}.yaml'
    yaml.safe_dump({'path': str(OUT), 'train': 'train/images', 'val': f'{split}/images', 'names': {0: 'fish'}}, open(y, 'w'))
    m = model.val(data=str(y), split='val', imgsz=IMGSZ, batch=BATCH, device=0, plots=False, verbose=False)
    return {'precision': round(float(m.box.mp), 3), 'recall': round(float(m.box.mr), 3),
            'mAP50': round(float(m.box.map50), 3), 'mAP50_95': round(float(m.box.map), 3)}

def gt_boxes(split, name):
    w, h = Image.open(OUT / split / 'images' / name).size
    rows = parse_yolo(OUT / split / 'labels' / (Path(name).stem + '.txt')) or []
    return np.array([[(xc - bw / 2) * w, (yc - bh / 2) * h, (xc + bw / 2) * w, (yc + bh / 2) * h]
                     for _, xc, yc, bw, bh in rows], np.float32).reshape(-1, 4)

def predict(split, names):
    preds = {}
    for i in range(0, len(names), 16):
        batch = [str(OUT / split / 'images' / n) for n in names[i:i + 16]]
        for n, r in zip(names[i:i + 16], model.predict(batch, conf=0.02, imgsz=IMGSZ, device=0, verbose=False)):
            preds[n] = (r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy())
    return preds

def iou(a, b):
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda z: (z[:, 2] - z[:, 0]) * (z[:, 3] - z[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)

def match(gt, boxes, confs, conf):   # greedy, highest confidence first, IoU >= 0.5 (project rule)
    keep = confs >= conf
    boxes, confs = boxes[keep], confs[keep]
    order = np.argsort(-confs)
    if len(gt) == 0:
        return 0, len(order), 0
    ious = iou(boxes, gt) if len(boxes) else np.zeros((0, len(gt)))
    used, tp = set(), 0
    for i in order:
        best_g, best_iou = -1, 0.5
        for g in range(len(gt)):
            if g not in used and ious[i, g] >= best_iou:
                best_g, best_iou = g, ious[i, g]
        if best_g >= 0:
            used.add(best_g); tp += 1
    return tp, len(order) - tp, len(gt) - len(used)

def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0; r = tp / (tp + fn) if tp + fn else 0.0
    return p, r, (2 * p * r / (p + r) if p + r else 0.0)

def per_image(split, names, gts, preds, conf):
    return {n: match(gts[n], *preds[n], conf) for n in names}

def total(per):
    return tuple(int(sum(v[k] for v in per.values())) for k in range(3))

def load(split, keep=lambda n: True):
    names = sorted(n for n in os.listdir(OUT / split / 'images') if keep(n))
    return names, {n: gt_boxes(split, n) for n in names}, predict(split, names)

is_trashcan = lambda n: n.startswith('vid_')
is_new = lambda n: n.startswith(('df_', 'br_'))
RESULTS = {'model': EVAL_WEIGHTS, 'imgsz': IMGSZ, 'run': RUN_NAME, 'matching': 'greedy IoU>=0.5, threshold chosen on val'}
rng = np.random.default_rng(0)
for tag, val_keep, test_split in (('trashcan', is_trashcan, 'test_trashcan'), ('new_datasets', is_new, 'test_new')):
    vn, vg, vp = load('val', val_keep)
    table = []
    for c in np.round(np.arange(0.05, 0.91, 0.05), 2):
        table.append((prf(*total(per_image('val', vn, vg, vp, c)))[2], float(c)))
    best_f1_val, conf = max(table)
    tn, tg, tp_ = load(test_split)
    per = per_image(test_split, tn, tg, tp_, conf)
    p, r, f1 = prf(*total(per))
    vids = collections.defaultdict(list)
    for n in tn:
        vids[GROUPS[f'{test_split}/{n}']].append(n)
    keys = list(vids); boot = []
    for _ in range(1000):
        sample = [n for k in rng.choice(keys, len(keys)) for n in vids[k]]
        boot.append(prf(*(sum(per[n][k] for n in sample) for k in range(3))))
    ci = lambda j: [round(float(np.percentile([b[j] for b in boot], q)), 3) for q in (2.5, 97.5)]
    tp, fp, fn = total(per)
    RESULTS[tag] = {
        'test_split': test_split, 'n_images': len(tn), 'n_videos': len(keys), 'n_fish': tp + fn,
        'conf_chosen_on_val': conf, 'val_f1_at_conf': round(best_f1_val, 3),
        'test': {'precision': round(p, 3), 'recall': round(r, 3), 'f1': round(f1, 3), 'tp': tp, 'fp': fp, 'fn': fn,
                 'precision_ci': ci(0), 'recall_ci': ci(1), 'f1_ci': ci(2)},
        'ultralytics_test': ul_metrics(test_split),
    }
    print(tag, json.dumps(RESULTS[tag], indent=1))

RESULTS['baseline_trashcan_test'] = {'model': 'old 16-class detector, conf 0.18', 'precision': 0.407, 'recall': 0.159, 'f1': 0.229}
json.dump(RESULTS, open(f'{DRIVE_DIR}/{TAG}_results.json', 'w'), indent=2)
print('\nTrashCan test F1: new', RESULTS['trashcan']['test']['f1'], RESULTS['trashcan']['test']['f1_ci'], 'vs baseline 0.229')
print('Saved', f'{DRIVE_DIR}/best_{TAG}.pt', 'and', f'{DRIVE_DIR}/{TAG}_results.json', '-> models/fish_detector/')

In [ ]:
# Image-level "fish present?" accuracy, same format as the coral model's image_level_coral_present
for tag, test_split in (('trashcan', 'test_trashcan'), ('new_datasets', 'test_new')):
    conf = RESULTS[tag]['conf_chosen_on_val']
    names, gts, preds = load(test_split)
    y = np.array([len(gts[n]) > 0 for n in names]); p = np.array([bool((preds[n][1] >= conf).any()) for n in names])
    tp = int((y & p).sum())
    RESULTS[tag]['image_level_fish_present'] = {
        'accuracy': round(float((y == p).mean()), 3), 'precision': round(tp / max(int(p.sum()), 1), 3),
        'recall': round(tp / max(int(y.sum()), 1), 3), 'n_positive': int(y.sum()), 'n_images': len(names),
        'always_majority_baseline': round(float(max(y.mean(), 1 - y.mean())), 3)}
    print(tag, RESULTS[tag]['image_level_fish_present'])
json.dump(RESULTS, open(f'{DRIVE_DIR}/{TAG}_results.json', 'w'), indent=2)
print('updated', f'{DRIVE_DIR}/{TAG}_results.json')

In [ ]:
FT = RESULTS

## 3. Keep the better one (chosen on validation F1, TrashCan)

In [ ]:
rows = [('as-is', ASIS)] + ([('fine-tuned', FT)] if FINETUNE_HOURS > 0 else [])
for name, r in rows:
    t = r['trashcan']
    print(f"{name:11s} val F1 {t['val_f1_at_conf']:.3f} | TrashCan TEST P {t['test']['precision']:.3f} R {t['test']['recall']:.3f} "
          f"F1 {t['test']['f1']:.3f} {t['test']['f1_ci']} mAP50 {t['ultralytics_test']['mAP50']:.3f} | DeepFish test F1 {r['new_datasets']['test']['f1']:.3f}")
keep = max(rows, key=lambda nr: nr[1]['trashcan']['val_f1_at_conf'])
print('\nKEEP:', keep[0], '->', keep[1]['model'], '| baseline TrashCan test F1 0.229 (old 16-class detector)')